# Regional Manager Weekly Briefing

**Generates a podcast-style weekly executive briefing for ONE region (passed in as `reg_id`) and writes a markdown file to OneLake under `RegionalManagerBriefing/`.**

- Calls the **Sales**, **Inventory**, and **Labor** Fabric Data Agents for region-scoped sections.
- Re-calls the Sales agent to synthesize a single 7–10 sentence Weekly Podcast narrative.
- No Azure OpenAI dependency — all language generation is done by the Fabric Data Agents.

**Scheduling:** orchestrate via a Fabric Data Pipeline that fans out per region (one notebook activity per `reg_id`) on a weekly cadence.

In [ ]:
# ── Install Fabric Data Agent SDK ──────────────────────────────────────────
%pip install fabric-data-agent-sdk -q

In [ ]:
# ── Parameters ──────────────────────────────────────────────────────────────
reg_id        = "REG000007"      # Region to brief (required)
week_ending   = "2026-01-17"     # YYYY-MM-DD; None → last Saturday
lookback_days = 7                # Briefing window
filepath      = "abfss://baade929-d8f1-41f5-978d-a3db577164e6@onelake.dfs.fabric.microsoft.com/a4f07204-b4e1-4007-b478-0cbcf7f31c5a/Files/retailstoreintelligence/weeklybriefing"               # OneLake base path for RegionalManagerBriefing/ (e.g. abfss://<ws>@onelake.dfs.fabric.microsoft.com/<lh>.Lakehouse/Files/RegionalManagerBriefing"               # OneLake base path for RegionalManagerBriefing/ (e.g. abfss://<ws>@onelake.dfs.fabric.microsoft.com/<lh>.Lakehouse/Files/RegionalManagerBriefing)

# Fabric Data Agent names (display names from Fabric portal → your workspace)
SALES_AGENT_NAME     = "Sales Agent"
INVENTORY_AGENT_NAME = "Inventory Agent"
LABOR_AGENT_NAME     = "Labor Agent"

In [ ]:
# ── Imports & date window ──────────────────────────────────────────────────
import json
from datetime import datetime, timedelta, date

from fabric.dataagent.client import FabricOpenAI

if not reg_id:
    raise ValueError("`reg_id` parameter is required.")
if not filepath:
    raise ValueError("`filepath` parameter is required (OneLake base path).")

if week_ending:
    period_end = datetime.strptime(week_ending, "%Y-%m-%d").date()
else:
    today = date.today()
    # Last Saturday (weekday 5)
    period_end = today - timedelta(days=(today.weekday() - 5) % 7 or 7)

period_start = period_end - timedelta(days=lookback_days - 1)
prior_end    = period_start - timedelta(days=1)
prior_start  = prior_end - timedelta(days=lookback_days - 1)
week_ending_str = period_end.strftime("%Y-%m-%d")

print(f"Region        : {reg_id}")
print(f"Week ending   : {week_ending_str}")
print(f"Period        : {period_start} → {period_end}")
print(f"Prior period  : {prior_start} → {prior_end}")

In [ ]:
# ── Region resolution ──────────────────────────────────────────────────────
region_df = spark.table("retail.regions") \
    .filter(f"reg_id = '{reg_id}'") \
    .select("reg_id", "reg_name") \
    .limit(1)

if region_df.count() == 0:
    raise ValueError(f"Region '{reg_id}' not found in retail.regions.")

row = region_df.first()
REG_ID   = row["reg_id"]
REG_NAME = row["reg_name"]
print(f"✓ Region resolved: {REG_NAME} ({REG_ID})")

In [ ]:
# ── Fabric Data Agent client (SDK) ────────────────────────────────────────
def _get_client(artifact_name: str) -> FabricOpenAI:
    return FabricOpenAI(artifact_name=artifact_name)

def call_agent(artifact_name: str, question: str) -> str:
    """Invoke a Fabric Data Agent via the SDK and return the assistant text reply.
    Errors are caught and returned as a string marker so the briefing still assembles."""
    if not artifact_name:
        return "_(agent call failed: artifact_name not configured)_"
    try:
        client = _get_client(artifact_name)
        assistant = client.beta.assistants.create(model="not-used")
        thread = client.beta.threads.create()
        client.beta.threads.messages.create(
            thread_id=thread.id, role="user", content=question
        )
        run = client.beta.threads.runs.create_and_poll(
            thread_id=thread.id, assistant_id=assistant.id
        )
        if run.status != "completed":
            return f"_(agent call failed: run status {run.status})_"

        messages = client.beta.threads.messages.list(thread_id=thread.id)
        for msg in messages.data:
            if msg.role == "assistant" and msg.content:
                return msg.content[0].text.value
        return "_(agent call failed: no assistant reply)_"
    except Exception as exc:
        return f"_(agent call failed: {exc})_"

print("✓ Fabric Data Agent client ready.")

In [ ]:
# ── Fetch the three domain markdown sections ───────────────────────────────────────────
scope = f"region {REG_NAME} ({REG_ID}), week ending {week_ending_str} (period {period_start} to {period_end}); prior week {prior_start} to {prior_end}"

sales_q = (
    f"For {scope}, return ONLY the markdown for these two subsections of a Weekly Performance Snapshot. "
    f"Do NOT include any podcast narrative or other sections.\n\n"
    f"### Key Metrics\n"
    f"- Weekly revenue and WoW%\n- Orders and WoW%\n- Gross margin % and avg basket size\n- MTD vs plan if available\n\n"
    f"### Department Read\n"
    f"- Top and bottom departments by revenue with WoW%\n- Notable brand highlights\n"
)

inv_q = (
    f"For {scope}, return ONLY the following markdown subsection of a Weekly Performance Snapshot. "
    f"Do NOT include any podcast narrative or other sections.\n\n"
    f"### Inventory Health\n"
    f"- OOS breakdown: TRUE_OOS · CHRONIC_OOS · PHANTOM_OOS · AT_RISK_OOS counts\n"
    f"- OSA rate vs prior week\n"
    f"- Top 3 lowest-OSA stores in the region\n"
    f"- Perishable lead-time risks if any\n"
)

labor_q = (
    f"For {scope}, return ONLY the following markdown subsection of a Weekly Performance Snapshot. "
    f"Do NOT include any podcast narrative or other sections.\n\n"
    f"### Labor Coverage\n"
    f"- Total headcount; understaffed stores vs store-type average (list names)\n"
    f"- Single-associate shift count with store names\n"
    f"- Avg orders/associate and tasks/associate\n"
)

print("→ Querying Sales agent…")
sales_md = call_agent(SALES_AGENT_NAME, sales_q)
print("→ Querying Inventory agent…")
inv_md   = call_agent(INVENTORY_AGENT_NAME, inv_q)
print("→ Querying Labor agent…")
labor_md = call_agent(LABOR_AGENT_NAME, labor_q)

errors = [name for name, md in [("sales", sales_md), ("inventory", inv_md), ("labor", labor_md)] if md.startswith("_(agent call failed")]
print(f"✓ Sections fetched. Errors: {errors or 'none'}")

In [ ]:
# ── Synthesis: re-call Sales agent to write the Weekly Podcast narrative ──────────────────────────
synth_q = (
    f"You are writing the WEEKLY PODCAST narrative section of a regional manager briefing for {scope}.\n\n"
    f"Use ONLY the three section outputs below as your source material. Do not invent numbers. "
    f"If a number is missing, speak qualitatively.\n\n"
    f"Write 7–10 sentences in a conversational, podcast-style tone as if a colleague is giving a verbal briefing to a regional manager. "
    f"Follow this arc: (1) one-sentence hook summarizing the week, (2) overall sales performance with WoW change, "
    f"(3) department-level trends — what is driving growth and what is soft, (4) transition to inventory — OSA rate and OOS classifications, "
    f"(5) staffing — headcount, coverage risks, productivity, (6) one or two cross-domain connections (e.g., a store with both low OSA and single-associate shifts), "
    f"(7) close with what to watch or act on next week.\n\n"
    f"Return ONLY the narrative paragraph(s). No headings, no bullets, no preamble.\n\n"
    f"--- SALES SECTION ---\n{sales_md}\n\n"
    f"--- INVENTORY SECTION ---\n{inv_md}\n\n"
    f"--- LABOR SECTION ---\n{labor_md}\n"
)

print("→ Synthesizing Weekly Podcast via Sales agent…")
podcast_md = call_agent(SALES_AGENT_NAME, synth_q)
if podcast_md.startswith("_(agent call failed"):
    errors.append("synthesis")
print("✓ Synthesis complete.")

In [ ]:
# ── Assemble final markdown ─────────────────────────────────────────────────────
generated_at = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S UTC")

briefing_md = f"""---
region_id: {REG_ID}
region_name: {REG_NAME}
week_ending: {week_ending_str}
generated_at: {generated_at}
---

# Weekly Executive Briefing — {REG_NAME}

**Region:** {REG_NAME} ({REG_ID})  
**Week ending:** {week_ending_str}  
**Period:** {period_start} → {period_end}

---

## Part 1 — Weekly Podcast

{podcast_md}

---

## Part 2 — Weekly Performance Snapshot

{sales_md}

{inv_md}

{labor_md}

---

_Prepared for Regional Manager — {REG_NAME}_  
_Generated {generated_at}_
"""

print(briefing_md[:1200] + ("\n…\n" if len(briefing_md) > 1200 else ""))

In [ ]:
# ── Write to OneLake ─────────────────────────────────────────────────────────
base = filepath.rstrip("/")
output_path = f"{base}/briefing_{REG_ID}_{week_ending_str}.md"

notebookutils.fs.put(output_path, briefing_md, True)

print(f"✓ Wrote briefing to: {output_path}")
print(f"  Region : {REG_NAME} ({REG_ID})")
print(f"  Week   : {week_ending_str}")
print(f"  Bytes  : {len(briefing_md)}")
print(f"  Errors : {errors or 'none'}")

notebookutils.notebook.exit(json.dumps({
    "reg_id": REG_ID, "week_ending": week_ending_str,
    "output_path": output_path, "errors": errors,
}))